# Context Pruning

### Source : TOOLS_CONTEXT_5/context_pruning_2.ipynb


### Pruning raw tool outputs with Provence - sentence-level semantic compression of tool results

- Given an agent with a deep message history approaching context capacity, use Provence to
  semantically prune tool outputs to only the sentences relevant to the current task


## Setup
A `.env` file with `DATABRICKS_HOST` and `DATABRICKS_TOKEN`

In [ ]:
import os
from dotenv import load_dotenv
import sys
sys.path.append(os.path.abspath(".."))
from _utils.sql_utils import sql_call

# Load credentials from .env at project root
load_dotenv()

# Verify the credentials are present
assert os.getenv("DATABRICKS_HOST"), "DATABRICKS_HOST not set — check your .env file"
assert os.getenv("DATABRICKS_TOKEN"), "DATABRICKS_TOKEN not set — check your .env file"
assert os.getenv("SQL_WAREHOUSE_ID"), "SQL_WAREHOUSE_ID not set — check your .env file"
assert os.getenv("MLFLOW_TRACKING_URI"), "MLFLOW_TRACKING_URI not set — check your .env file"

## Imports and config

In [ ]:
import json
import uuid
import mlflow
import tiktoken
from databricks.sdk import WorkspaceClient
from databricks_langchain import ChatDatabricks, UCFunctionToolkit
from langchain_core.messages import (
    AnyMessage, AIMessage, HumanMessage, SystemMessage, ToolMessage
)

In [ ]:
# catalog, schema, and SQL warehouse id
CATALOG      = "demo"
SCHEMA       = "context"
WAREHOUSE_ID = os.getenv("SQL_WAREHOUSE_ID")   

# Provence: skip pruning for tool results smaller than this (no overhead for tiny payloads)
SMALL_RESULT_THRESHOLD = 100          # tokens
MODEL_CONTEXT_WINDOW   = 128_000      # meta-llama-3-3-70b context length

In [ ]:
w   = WorkspaceClient()
llm = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct")
enc = tiktoken.get_encoding("cl100k_base")

mlflow.set_experiment("/Users/oliver@mlops-media.com/context_pruning_3")
print(f"Workspace host: {w.config.host}")

---
# Pruning raw tool outputs with Provence

## 1 - Simulate a multi-turn agent with a filling context window

We build a realistic 6-turn message history with tool results of varying size:
some are small JSON objects, others are large table scans.


In [ ]:
def make_large_payload(label: str, rows: int = 200) -> str:
    """Simulate a bulky tool result such as a table scan or large retrieval chunk."""
    return json.dumps({
        "source":    label,
        "row_count": rows,
        "data":      [{"id": i, "value": f"row_{i}_{label}"} for i in range(rows)]
    })

def count_message_tokens(messages: list) -> int:
    return sum(len(enc.encode(m.content or "")) for m in messages if hasattr(m, "content"))

In [ ]:
deep_history = [
    # Turn 1 - large product catalog
    HumanMessage(content="Get me the full product catalog"),
    AIMessage(content="", tool_calls=[{"id": "t1", "name": "get_product_catalog", "args": {}}]),
    ToolMessage(content=make_large_payload("product_catalog", rows=300), tool_call_id="t1"),
    AIMessage(content="The catalog contains 300 items."),
    # Turn 2 - large revenue report
    HumanMessage(content="Now get Q1 revenue by region"),
    AIMessage(content="", tool_calls=[{"id": "t2", "name": "run_revenue_report", "args": {"period": "Q1"}}]),
    ToolMessage(content=make_large_payload("revenue_report_q1", rows=50), tool_call_id="t2"),
    AIMessage(content="Q1 revenue report retrieved. I have the regional breakdown."),
    # Turn 3 - small account lookup
    HumanMessage(content="What plan is customer C-1042 on?"),
    AIMessage(content="", tool_calls=[{"id": "t3", "name": "lookup_account_details_v2", "args": {"account_id": "C-1042"}}]),
    ToolMessage(content=json.dumps({"account_id": "C-1042", "plan": "enterprise", "billing_email": "c1042@corp.com"}), tool_call_id="t3"),
    AIMessage(content="Customer C-1042 is on the enterprise plan."),
    # Turn 4 - large support docs retrieval
    HumanMessage(content="Search for support docs about login errors"),
    AIMessage(content="", tool_calls=[{"id": "t4", "name": "ai_search", "args": {"query": "login errors"}}]),
    ToolMessage(content=make_large_payload("support_docs_login", rows=100), tool_call_id="t4"),
    AIMessage(content="Found 100 support documents related to login errors."),
    # Turn 5 - small discount calc
    HumanMessage(content="What is the discount for customer C-1042 on a 500-unit order?"),
    AIMessage(content="", tool_calls=[{"id": "t5", "name": "calculate_discount", "args": {"account_id": "C-1042", "volume": 500}}]),
    ToolMessage(content=json.dumps({"discount_pct": 12.5}), tool_call_id="t5"),
    AIMessage(content="The discount for C-1042 on 500 units is 12.5%."),
    # Turn 6 - current turn
    HumanMessage(content="Now create a quote for customer C-1042 for 500 units of SKU-007"),
]


In [ ]:
total_tokens = count_message_tokens(deep_history)
pct_used     = 100 * total_tokens / MODEL_CONTEXT_WINDOW

print(f"Message history : {len(deep_history)} messages")
print(f"Total tokens    : ~{total_tokens:,}")
print(f"Context usage   : {pct_used:.1f}% of {MODEL_CONTEXT_WINDOW:,}-token window")
if pct_used >= 75:
    print("WARNING: Context above 75% -- pruning recommended before next turn")


## 2 - Sentence-level pruning with Provence

Instead of heuristic rules (size, age), we use **Provence** (`naver/provence-reranker-debertav3-v1`) --
a 430M-parameter DeBERTa-based model from Naver Labs Europe trained for context pruning in RAG pipelines.

**How it works:**
- Input: a *question* (the current agent task) + a *context passage* (the tool result)
- Output: a pruned passage keeping only sentences relevant to the question, plus a relevance score
- Architecture: sequence labelling -- each sentence gets a binary keep/discard label in a single forward pass
- No threshold tuning required: Provence automatically detects how many sentences to retain


In [ ]:
from transformers import AutoModel
# Load Provence
# Runs on CPU for moderate-sized tool results; faster on a GPU-enabled cluster
provence = AutoModel.from_pretrained(
    "naver/provence-reranker-debertav3-v1",
    trust_remote_code=True,
)
print("Provence loaded")


## 3 - Define the Provence pruning functions

In [ ]:
def prune_tool_result_with_provence(
    query: str,
    tool_result: str,
    min_tokens: int = SMALL_RESULT_THRESHOLD,
) -> dict:
    """
    Use Provence to prune a single tool result to only the sentences
    relevant to the current query.

    Args:
        query       : The current user question / agent task.
        tool_result : Raw tool output as a string (JSON or plain text).
        min_tokens  : Results smaller than this are returned as-is.

    Returns:
        dict with keys:
            pruned_content   : Pruned text (or original if too small)
            reranking_score  : Provence relevance score for the whole passage
            tokens_before    : Token count before pruning
            tokens_after     : Token count after pruning
            was_pruned       : Whether Provence modified the content
    """
    tokens_before = len(enc.encode(tool_result))

    # Skip Provence for small results -- not worth the overhead
    if tokens_before < min_tokens:
        return {
            "pruned_content":  tool_result,
            "reranking_score": None,
            "tokens_before":   tokens_before,
            "tokens_after":    tokens_before,
            "was_pruned":      False,
        }

    # Provence expects plain text -- convert JSON to readable sentences
    try:
        parsed = json.loads(tool_result)
        if isinstance(parsed, dict):
            context_text = "\n".join(f"{k}: {v}" for k, v in parsed.items() if k != "data")
            if "data" in parsed:
                n = len(parsed["data"])
                context_text += f"\ndata: contains {n} rows from source '{parsed.get('source', 'unknown')}'"
        elif isinstance(parsed, list):
            context_text = "\n".join(str(item) for item in parsed[:20])
        else:
            context_text = str(parsed)
    except (json.JSONDecodeError, TypeError):
        context_text = tool_result

    result          = provence.process(query, context_text)
    pruned_content  = result.get("pruned_context", context_text)
    reranking_score = result.get("reranking_score", None)
    tokens_after    = len(enc.encode(pruned_content))

    return {
        "pruned_content":  pruned_content,
        "reranking_score": reranking_score,
        "tokens_before":   tokens_before,
        "tokens_after":    tokens_after,
        "was_pruned":      pruned_content != context_text,
    }


In [ ]:
def prune_history_with_provence(
    messages: list,
    current_query: str,
) -> tuple[list, dict]:
    """
    Apply Provence pruning to every ToolMessage in a message history.

    Args:
        messages      : Full message list.
        current_query : The current user task -- used as the Provence question.

    Returns:
        (pruned_messages, stats_dict)
    """
    pruned = list(messages)
    stats  = {"tools_processed": 0, "tools_pruned": 0,
               "tokens_before": 0,  "tokens_after": 0}

    tool_names = {}
    for m in messages:
        if isinstance(m, AIMessage) and getattr(m, "tool_calls", None):
            for tc in m.tool_calls:
                tool_names[tc["id"]] = tc["name"]

    for i, msg in enumerate(messages):
        if not isinstance(msg, ToolMessage):
            continue

        tool_name = tool_names.get(msg.tool_call_id, "unknown")
        result    = prune_tool_result_with_provence(current_query, msg.content or "")

        stats["tools_processed"] += 1
        stats["tokens_before"]   += result["tokens_before"]
        stats["tokens_after"]    += result["tokens_after"]

        if result["was_pruned"]:
            stats["tools_pruned"] += 1
            pruned[i] = ToolMessage(
                content=result["pruned_content"],
                tool_call_id=msg.tool_call_id,
            )
            print(f"  [{tool_name}] {result['tokens_before']:,} -> {result['tokens_after']:,} tokens "
                  f"(score: {result['reranking_score']:.3f})")
        else:
            reason = "too small" if result["tokens_before"] < SMALL_RESULT_THRESHOLD else "already dense"
            print(f"  [{tool_name}] kept as-is ({result['tokens_before']:,} tokens -- {reason})")

    return pruned, stats


## 4 - Apply Provence to the message history and measure the saving

In [ ]:
import nltk
nltk.download('punkt_tab')

current_query = "Create a quote for customer C-1042 for 500 units of SKU-007"

print(f"Current query: {current_query}")
print()
print("=== Provence pruning -- per tool result ===")
pruned_history, prune_stats = prune_history_with_provence(deep_history, current_query)

tokens_before = count_message_tokens(deep_history)
tokens_after  = count_message_tokens(pruned_history)
reduction_pct = 100 * (1 - tokens_after / max(tokens_before, 1))

print()
print("=== Provence pruning results ===")
print(f"  Tool results processed : {prune_stats['tools_processed']}")
print(f"  Tool results pruned    : {prune_stats['tools_pruned']}")
print(f"  Tokens before          : ~{count_message_tokens(deep_history):,}")
print(f"  Tokens after           : ~{tokens_after:,}")
print(f"  Reduction              : {reduction_pct:.0f}%")


In [ ]:
print("=== Provence pruned content ===")
for message in pruned_history :
    print(message.content)